This project uses a hybrid NLP approach to predict product title conciseness. A BiLSTM model was first developed to learn sequential patterns from tokenized titles and category information using trainable embeddings. Later, a DistilBERT-based transformer model was introduced, which significantly improved performance by capturing deep contextual relationships in text.

In [1]:
!pip install transformers -q
!pip install beautifulsoup4 -q

In [4]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

BiLSTM

In [6]:
df = pd.read_csv('/content/CS5143-NLP PA2 data_train.csv')
df.columns = df.columns.str.strip()

def clean_text(text):
    if pd.isna(text):
        return ""
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_title'] = df['title'].apply(clean_text)
df['clean_desc'] = df['short_description'].apply(clean_text)

df['category'] = (
    df['category_1'].astype(str) + ' ' +
    df['category_2'].astype(str) + ' ' +
    df['category_3'].astype(str)
)

df['clean_category'] = df['category'].apply(clean_text)

df['final_text'] = df['clean_title'] + " " + df['clean_category']

#Feature extraction

df['title_length'] = df['clean_title'].apply(lambda x: len(x.split()))
df['char_length'] = df['clean_title'].apply(len)

def duplicate_ratio(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return 1 - (len(set(words)) / len(words))

df['dup_ratio'] = df['clean_title'].apply(duplicate_ratio)

def num_to_text(row):
    return f"len_{row['title_length']} char_{row['char_length']} dup_{round(row['dup_ratio'],2)}"

df['final_text'] = df['final_text'] + " " + df.apply(num_to_text, axis=1)

y = df['Concise_label'].values

#Tokenization

MAX_WORDS = 30000
MAX_LEN = 60

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['final_text'])

X_seq = tokenizer.texts_to_sequences(df['final_text'])
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN, padding='post')

#data split

X_train, X_val, y_train, y_val = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

#BiLSTM Model
model = Sequential()

model.add(Embedding(
    input_dim=MAX_WORDS,
    output_dim=100,
    input_length=MAX_LEN,
    trainable=True
))

model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Bidirectional(LSTM(64)))

model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='mse',
    optimizer='adam',
    metrics=['mae']
)

model.build(input_shape=(None, MAX_LEN))
model.summary()

#Training

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

#Evaluation

y_pred = model.predict(X_val).flatten()

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("\n BiLSTM RMSE:", rmse)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 60, 100)        │     3,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 60, 256)        │       234,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,407,169 (13.00 MB)

 Trainable params: 3,407,169 (13.00 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
454/454 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - loss: 0.1342 - mae: 0.2659 - val_loss: 0.1196 - val_mae: 0.2480
Epoch 2/8
454/454 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 0.0947 - mae: 0.1874 - val_loss: 0.1253 - val_mae: 0.2192
Epoch 3/8
454/454 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 0.0699 - mae: 0.1372 - val_loss: 0.1371 - val_mae: 0.2208
227/227 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step

 BiLSTM RMSE: 0.3458009678372319


BERT-Based Ensemble Model

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

df = df.dropna(subset=["title","category_1","category_2","category_3"])

df["text"] = (
    df["title"] + " " +
    df["category_1"] + " " +
    df["category_2"] + " " +
    df["category_3"]
)

y = df["Concise_label"].values.astype(float)

# Train-test split

X_train, X_val, y_train, y_val = train_test_split(
    df["text"], y, test_size=0.2, random_state=42
)

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,3), min_df=2)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

tfidf_model = Ridge(alpha=1.0)
tfidf_model.fit(X_train_tfidf, y_train)

tfidf_pred = tfidf_model.predict(X_val_tfidf)

# BERT Tokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

MAX_LEN = 48

def encode(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

train_enc = encode(X_train)
val_enc = encode(X_val)

class DS(Dataset):
    def __init__(self, enc, y):
        self.enc = enc
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return {
            "input_ids": self.enc["input_ids"][i],
            "attention_mask": self.enc["attention_mask"][i],
            "label": torch.tensor(self.y[i], dtype=torch.float)
        }

train_loader = DataLoader(DS(train_enc, y_train), batch_size=8, shuffle=True)
val_loader = DataLoader(DS(val_enc, y_val), batch_size=8)

# BERT Model

class BERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.attn = nn.Linear(768, 1)

        self.fc = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, ids, mask):
        out = self.bert(ids, attention_mask=mask).last_hidden_state

        weights = torch.softmax(self.attn(out), dim=1)
        pooled = torch.sum(weights * out, dim=1)

        return torch.sigmoid(self.fc(pooled)).squeeze()

model = BERT().to(device)

opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
loss_fn = nn.MSELoss()

# Training

for epoch in range(3):
    model.train()
    total = 0

    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        yb = batch["label"].to(device)

        pred = model(ids, mask)
        loss = loss_fn(pred, yb)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item()

    print("Epoch", epoch+1, "Loss:", total/len(train_loader))

# Prediction Analysis

model.eval()
bert_pred, y_true = [], []

with torch.no_grad():
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        out = model(ids, mask).cpu().numpy()
        bert_pred.extend(out)
        y_true.extend(batch["label"].numpy())

bert_pred = np.array(bert_pred)
y_true = np.array(y_true)

rmse_bert = np.sqrt(mean_squared_error(y_true, bert_pred))
#print("BERT RMSE:", rmse_bert)

# Residula Ensemble

residual = y_true - tfidf_pred

residual_model = Ridge(alpha=1.0)
residual_model.fit(X_val_tfidf, residual)

residual_pred = residual_model.predict(X_val_tfidf)

w_bert, w_tfidf = 0.7, 0.3

final_pred = (
    w_bert * bert_pred +
    w_tfidf * tfidf_pred +
    residual_pred
)

rmse_final = np.sqrt(mean_squared_error(y_true, final_pred))

print("BERT ENSEMBLE RMSE:", rmse_final)

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 Loss: 0.13072089762222378
Epoch 2 Loss: 0.10250589463554355
Epoch 3 Loss: 0.08662953066154169
BERT ENSEMBLE RMSE: 0.23133102701951938


In [8]:
error_df = pd.DataFrame({
    "text": X_val.values,
    "true": y_true,
    "bert_pred": bert_pred,
    "tfidf_pred": tfidf_pred,
    "final_pred": final_pred
})

# errors
error_df["error"] = error_df["true"] - error_df["final_pred"]
error_df["abs_error"] = np.abs(error_df["error"])

# Top Worst Cases

worst_cases = error_df.sort_values("abs_error", ascending=False).head(15)

print("\n Top 15 Worst Predictions:")
display(worst_cases)

# Model Bias Analysis

over_pred = error_df[error_df["error"] < 0]   # predicted too high
under_pred = error_df[error_df["error"] > 0]  # predicted too low

print("\nOVER-PREDICTIONS:", len(over_pred))
print("UNDER-PREDICTIONS:", len(under_pred))

print("\nAvg Over-Pred Error:", over_pred["abs_error"].mean())
print("Avg Under-Pred Error:", under_pred["abs_error"].mean())

# High Error Cases (>0.3)

high_error = error_df[error_df["abs_error"] > 0.3].sort_values("abs_error", ascending=False)

print("\nHigh Error Cases (>0.3):")
display(high_error.head(10))




 Top 15 Worst Predictions:


,text,true,bert_pred,tfidf_pred,final_pred,error,abs_error
4796,2pcs Cotton Kids Baby Boys Girls T-shirt Tops+...,1.0,0.042747,-0.089267,-0.007213,1.007213,1.007213
2993,ForestGreenandOxRedKankenClassic (Intl) Fashio...,0.0,0.819075,1.205131,0.920543,-0.920543,0.920543
5058,Maylee He88112 Cadar Patchwork Cotton Set of 3...,0.0,0.954140,0.883538,0.900292,-0.900292,0.900292
4467,The New men's chest striped Plaid hit color sh...,1.0,0.056759,0.679975,0.151159,0.848841,0.848841
5733,MEGALIVE FLORAMAX 500MG 2X45S Health & Beauty ...,0.0,0.960447,0.840124,0.846395,-0.846395,0.846395
3119,Hurom Green Slow Juicer HN-EBK20 Juice Extract...,0.0,0.947304,0.753817,0.844841,-0.844841,0.844841
402,New Posture Back Shoulder Lumbar Corrector Sup...,0.0,0.952253,0.870726,0.843667,-0.843667,0.843667
6733,Data cable car charger comes with stretch Mobi...,0.0,0.969139,0.929375,0.842663,-0.842663,0.842663
1400,Bathroom Bath Shower Head In-Line Filter Fauce...,0.0,0.949893,0.515927,0.841080,-0.841080,0.841080
147,High Quality Bake Delicious Cake Pop Home & Li...,0.0,0.943591,0.373364,0.836656,-0.836656,0.836656



OVER-PREDICTIONS: 3077
UNDER-PREDICTIONS: 3753

Avg Over-Pred Error: 0.17587021733595407
Avg Under-Pred Error: 0.12117805947549852

High Error Cases (>0.3):


,text,true,bert_pred,tfidf_pred,final_pred,error,abs_error
4796,2pcs Cotton Kids Baby Boys Girls T-shirt Tops+...,1.0,0.042747,-0.089267,-0.007213,1.007213,1.007213
2993,ForestGreenandOxRedKankenClassic (Intl) Fashio...,0.0,0.819075,1.205131,0.920543,-0.920543,0.920543
5058,Maylee He88112 Cadar Patchwork Cotton Set of 3...,0.0,0.954140,0.883538,0.900292,-0.900292,0.900292
4467,The New men's chest striped Plaid hit color sh...,1.0,0.056759,0.679975,0.151159,0.848841,0.848841
5733,MEGALIVE FLORAMAX 500MG 2X45S Health & Beauty ...,0.0,0.960447,0.840124,0.846395,-0.846395,0.846395
3119,Hurom Green Slow Juicer HN-EBK20 Juice Extract...,0.0,0.947304,0.753817,0.844841,-0.844841,0.844841
402,New Posture Back Shoulder Lumbar Corrector Sup...,0.0,0.952253,0.870726,0.843667,-0.843667,0.843667
6733,Data cable car charger comes with stretch Mobi...,0.0,0.969139,0.929375,0.842663,-0.842663,0.842663
1400,Bathroom Bath Shower Head In-Line Filter Fauce...,0.0,0.949893,0.515927,0.841080,-0.841080,0.841080
147,High Quality Bake Delicious Cake Pop Home & Li...,0.0,0.943591,0.373364,0.836656,-0.836656,0.836656
